# Lazy Browse — DestinE Climate DT Portfolio

This notebook provides an **instant xarray view** of DestinE Climate DT Generation 2 data.
Variables and coordinates appear immediately, actual data is fetched from the data bridge **only when you access values** (e.g. plotting, `.values`, `.compute()`).

**Prerequisites:** run `01_key_destine_once.ipynb` once to authenticate.

In [ ]:
import logging, warnings
import earthkit.data

# Disable earthkit disk cache (polytope_zarr caches decoded arrays in memory)
earthkit.data.config.set("cache-policy", "off")

# Silence verbose output from polytope / earthkit internals
for _ln in ("polytope", "polytope.api", "earthkit.data", "urllib3"):
    logging.getLogger(_ln).setLevel(logging.WARNING)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
import numpy as np
import pandas as pd
from polytope_zarr import PolytopeZarrStore
from destine_portfolio import PORTFOLIO_GEN2_CLMN

## 1. Define the dataset

We declare which coordinates, variables, and Polytope request fields to use.
Nothing is downloaded here — the store just builds metadata.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────
MODELS       = ["ICON", "IFS-FESOM", "IFS-NEMO"]
EXPERIMENT   = "hist"              # 'hist', 'SSP3-7.0'
YEARS        = range(1990, 2015)   # 25-year historical period

# ── Choose a levtype (uncomment one) ──────────────────────────────
LEVTYPE = "sfc"                    # 34 vars — surface atmosphere
# LEVTYPE = "pl"                   #  9 vars — pressure levels (19 levels, 1000–1 hPa)
# LEVTYPE = "hl"                   #  2 vars — height levels (100 m, IFS-only)
# LEVTYPE = "sol"                  #  2 vars — soil / snow (5 levels)
# LEVTYPE = "o2d"                  # 13 vars — 2-D ocean & sea ice
# LEVTYPE = "o3d"                  #  5 vars — 3-D ocean (up to 75 levels)

store = PolytopeZarrStore.from_climate_dt(
    models=MODELS,
    experiment=EXPERIMENT,
    levtype=LEVTYPE,
    years=YEARS,
)
print(store)

## 2. Open as xarray Dataset

This is **instant** — no data downloaded yet. You see all variables,
dimensions, and coordinate values.

In [ ]:
ds = store.open()
ds

## 3. Plot a single monthly field (triggers lazy fetch)

Only now does the store actually call Polytope — fetching data for the
selected model, time, and variable.

In [ ]:
import healpy as hp
import matplotlib.pyplot as plt

field = ds["avg_2t"].sel(model="IFS-FESOM", time="2010-06-01")
print(f"Fetching {dict(field.sizes)} values...")  # triggers the Polytope request

hp.mollview(field.values, title="ICON — avg 2m temperature — Jun 2010",
            unit="K", cmap="RdYlBu_r", nest=True, flip='geo')
plt.show()

In [ ]:
#annual mean
field2 = ds["avg_tprate"].sel(model="IFS-FESOM", time=slice("2014-01", "2014-12")).mean("time")

hp.mollview(field2.values,
            title="IFS-FESOM — avg total precipitation rate — year 2014",
            unit="m", cmap="Blues", min=0, max=0.0001, nest=True, flip='geo')
plt.show()

## 4. Compare experiments: climate change signal

Create a second store for the scenario experiment, then take the difference to the historical.
`.polytope.sel()` automatically sets `batch_years` from the time slice so each call fetches all requested years in one Polytope request.

In [ ]:
SCEN_EXPERIMENT = "SSP3-7.0"
SCEN_YEARS      = range(2015, 2050)

scen_store = PolytopeZarrStore.from_climate_dt(
    models=MODELS,
    experiment=SCEN_EXPERIMENT,
    levtype=LEVTYPE,
    years=SCEN_YEARS,
)

ds_scen = scen_store.open()
ds_scen

In [ ]:
# .polytope.sel() auto-sets the number of batched years from the time slice -> single request per period, instead of ~30 requests per period
hist_field = ds["avg_2t"].polytope.sel(model="IFS-FESOM", time=slice("1990-01", "2014-12"))
scen_field = ds_scen["avg_2t"].polytope.sel(model="IFS-FESOM", time=slice("2015-01", "2049-12"))

diff = scen_field.mean("time") - hist_field.mean("time")

hp.mollview(diff,
            title="IFS-FESOM — ΔT (SSP3-7.0 minus historical)",
            unit="K", cmap="RdBu_r", min=-3, max=3, nest=True, flip='geo')
plt.show()

## 5. Test, not working yet: Server-side spatial subsetting (Polytope features), USE TEST NOTEBOOKS INSTEAD

`.polytope.sel()` also supports **server-side spatial extraction** via the
Polytope feature API.  Pass `bbox`, `polygon`, or `point` to get data on a
regular lat/lon grid instead of HEALPix cells.

In [ ]:
# Bounding box over Central Europe — single date, returns lat/lon grid
bbox_ds = ds["avg_2t"].polytope.sel(
    model="IFS-FESOM",
    time="2010-06",
    bbox=(47, 5, 55, 15),   # (south, west, north, east)
)
bbox_ds

In [ ]:
# Timeseries at a single point (Berlin) over one year
ts_ds = ds["avg_2t"].polytope.sel(
    model="IFS-FESOM",
    time=slice("2010-01", "2010-12"),
    point=(52.5, 13.4),  # (lat, lon) — Berlin
)
ts_ds

## Notes for further use

- **Automatic batching:** use `.polytope.sel(time=slice(...))` to auto-batch Polytope requests over the requested year range. Regular `.sel()` fetches one year at a time (12 months per request).
- Try the other `LEVTYPES` as well in order to browse the full Climate DT portfolio, i.e. `o2d/o3d`, `pl/hl`, or `sol`
- `store.clear_cache()` frees memory from previously fetched fields.

In [ ]:
# Free memory if needed
store.clear_cache()
scen_store.clear_cache()